# ML Intent Analysis

This notebook trains and demos a customer-support **intent classifier**. It
maps a free-text support message to one of **77 banking support intents** using
classic NLP (TF-IDF) plus Logistic Regression, trained on the
[PolyAI banking support dataset](https://github.com/PolyAI-LDN/task-specific-datasets).

It walks through the full lifecycle: load → analyse → baseline model → error
analysis → model/feature/regularization experiments → final evaluation → save →
explain a prediction. Project-level docs (setup, repo layout, usage) are in the
`README.md`.

Shared setup (imports, data loading, train/validation split) lives in the first
cells; each later step only adds what is new.

### Model used here

| Stage      | Component                                       |
| ---------- | ----------------------------------------------- |
| Features   | `TfidfVectorizer(lowercase=True, ngram=(1,1))`  |
| Classifier | `LogisticRegression(C=10.0, max_iter=1000)`     |
| Task       | Multi-class text classification (77 intents)    |

Evaluation uses **accuracy** and **macro-F1** to account for class imbalance.

---

## Notebook workflow

1. Load the data
2. Analyse the data
3. Build a baseline model
4. Analyse errors
5. Compare models
6. Compare feature sets (n-grams)
7. Tune regularization
8. Final evaluation on the test set
9. Train and save the final model
10. Explain a single prediction

## 1. Load the data

In [1]:
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

base_url = (
    "https://raw.githubusercontent.com/"
    "PolyAI-LDN/task-specific-datasets/master/banking_data/"
)

train_df = pd.read_csv(base_url + "train.csv")
test_df = pd.read_csv(base_url + "test.csv")

print("Training examples:", len(train_df))
print("Test examples:", len(test_df))

print("\nTraining columns:", train_df.columns.tolist())
print("\nFirst training example:")
print(train_df.iloc[0])

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)

Training examples: 10003
Test examples: 3080

Training columns: ['text', 'category']

First training example:
text        I am still waiting on my card?
category                      card_arrival
Name: 0, dtype: str


## 2. Analyse the data

In [2]:
print("Dataset shape:", train_df.shape)

print("Missing Values:")
print(train_df.isnull().sum())

print("\nNumber of unique values:")
print(train_df["category"].nunique())

print("\nExamples from unique categories:")
print(train_df.groupby("category")["text"].first().head(10))

print("\nMost common categories:")
print(train_df["category"].value_counts().head(10))

Dataset shape: (10003, 2)
Missing Values:
text        0
category    0
dtype: int64

Number of unique values:
77

Examples from unique categories:
category
Refund_not_showing_up                               I don't see my refund money yet in my account....
activate_my_card                                     Please help me with my card.  It won't activate.
age_limit                                                   I want to open an account for my children
apple_pay_or_google_pay                             Hi, I have an apple watch. How do I use it to ...
atm_support                                                        I can use this card at which ATMs?
automatic_top_up                                    Can I add money automatically to my account wh...
balance_not_updated_after_bank_transfer                                 help my money isnt showing up
balance_not_updated_after_cheque_or_cash_deposit        Why is my last cheque deposit taking so long?
beneficiary_not_allowed      

## 3. Build a baseline model

In [3]:
X = train_df["text"]
y = train_df["category"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_val)

accuracy = accuracy_score(y_val, predictions)
macro_f1 = f1_score(y_val, predictions, average="macro")

print(f"Training examples: {len(X_train)}")
print(f"Validation examples: {len(X_val)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")

print("\nSample predictions:")
for text, actual, predicted in zip(
    X_val.iloc[:10],
    y_val.iloc[:10],
    predictions[:10]
):
    print(f"\nText: {text}")
    print(f"Actual: {actual}")
    print(f"Predicted: {predicted}")

Training examples: 8002
Validation examples: 2001
Accuracy: 0.8431
Macro-F1: 0.8389

Sample predictions:

Text: The app failed to verify my id.
Actual: unable_to_verify_identity
Predicted: unable_to_verify_identity

Text: What are the fees imposed when I make a transaction?
Actual: card_payment_fee_charged
Predicted: cancel_transfer

Text: Can you tell me why I haven't received my new card yet?
Actual: card_arrival
Predicted: card_arrival

Text: I made a purchase with my card and I was charged a fee for using it
Actual: card_payment_fee_charged
Predicted: card_payment_fee_charged

Text: Why is there a verification to identity
Actual: why_verify_identity
Predicted: why_verify_identity

Text: Hi, I am unable to see transaction in my account which i made couples of hours ago from my UK account. Please help me in this.
Actual: balance_not_updated_after_bank_transfer
Predicted: balance_not_updated_after_bank_transfer

Text: How long do I have  to wait for my transaction to be completed?
Act

## 4. Analyse errors

Reuses the baseline `model` and `predictions` from step 3.

In [4]:
report = classification_report(y_val, predictions, output_dict=True, zero_division=0)

metrics = pd.DataFrame(report).T
class_metrics = metrics.loc[metrics.index.isin(y.unique())]

print("\n10 categories with the lowest F1:")
print(
    class_metrics
    .sort_values("f1-score")[["precision", "recall", "f1-score", "support"]]
    .head(10)
    .round(3)
    .to_string()
)

errors = pd.DataFrame({
    "text": X_val.to_numpy(),
    "actual": y_val.to_numpy(),
    "predicted": predictions,
})

errors = errors[errors["actual"] != errors["predicted"]]

print(f"\nIncorrect predictions: {len(errors)}")
print("\nFirst 15 errors:")
print(errors.head(15).to_string(index=False))

errors.to_csv("validation_errors.csv", index=False)
print("\nSaved errors to validation_errors.csv")

confusion_pairs = (
    errors.groupby(["actual", "predicted"])
    .size()
    .sort_values(ascending=False)
    .rename("count")
    .reset_index()
)

print("\nTop 15 confusion pairs:")
print(confusion_pairs.head(15).to_string(index=False))


10 categories with the lowest F1:
                                    precision  recall  f1-score  support
virtual_card_not_working                1.000   0.375     0.545      8.0
card_acceptance                         0.778   0.583     0.667     12.0
get_disposable_virtual_card             0.750   0.632     0.686     19.0
top_up_failed                           0.783   0.621     0.692     29.0
transfer_not_received_by_recipient      0.686   0.706     0.696     34.0
topping_up_by_card                      0.923   0.571     0.706     21.0
lost_or_stolen_card                     0.733   0.688     0.710     16.0
declined_card_payment                   0.676   0.806     0.735     31.0
order_physical_card                     0.773   0.708     0.739     24.0
declined_cash_withdrawal                0.718   0.800     0.757     35.0

Incorrect predictions: 314

First 15 errors:
                                                                               text                             actu

## 5. Compare models

In [5]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "decision_tree": DecisionTreeClassifier(max_depth=30, min_samples_leaf=2, random_state=42),
}

result = []

for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
        ("clf", classifier),
    ])

    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = time.perf_counter() - start_time

    predictions = pipeline.predict(X_val)
    accuracy = accuracy_score(y_val, predictions)
    f1 = f1_score(y_val, predictions, average="weighted")

    result.append({
        "model": name,
        "train_time": train_seconds,
        "accuracy": accuracy,
        "f1_score": f1,
    })

    print(f"\n{name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro-F1: {f1:.4f}")
    print(f"Training time: {train_seconds:.2f} seconds")

print("\n=== Model comparison ===")
print(
    pd.DataFrame(result)
    .sort_values("f1_score", ascending=False)
    .round(4)
    .to_string(index=False)
)


logistic_regression
Accuracy: 0.8431
Macro-F1: 0.8422
Training time: 28.66 seconds

decision_tree
Accuracy: 0.5442
Macro-F1: 0.5953
Training time: 1.43 seconds

=== Model comparison ===
              model  train_time  accuracy  f1_score
logistic_regression     28.6554    0.8431    0.8422
      decision_tree      1.4345    0.5442    0.5953


## 6. Compare feature sets (n-grams)

In [6]:
experiment_results = {
    "Unigram": (1, 1),
    "Unigram + Bigram": (1, 2),
}

results = []

for name, ngram_range in experiment_results.items():
    model = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram_range, max_features=30000)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ])

    start = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = time.perf_counter() - start

    predictions = model.predict(X_val)

    results.append({
        "Features": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro-F1": f1_score(y_val, predictions, average="macro"),
        "Training seconds": train_seconds,
    })

print(pd.DataFrame(results).round(4).to_string(index=False))

        Features  Accuracy  Macro-F1  Training seconds
         Unigram    0.8626    0.8619            6.6168
Unigram + Bigram    0.8431    0.8389           19.5464


## 7. Tune regularization strength (C)

In [7]:
results = []

for c_value in [0.1, 1.0, 10.0]:
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 1),
            max_features=30000,
        )),
        ("classifier", LogisticRegression(
            C=c_value,
            max_iter=1000,
            random_state=42,
        )),
    ])

    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    predictions = model.predict(X_val)

    results.append({
        "C": c_value,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro-F1": f1_score(y_val, predictions, average="macro"),
        "Training seconds": elapsed,
    })

print(pd.DataFrame(results).round(4).to_string(index=False))

   C  Accuracy  Macro-F1  Training seconds
 0.1    0.7221    0.6689            2.5541
 1.0    0.8626    0.8619            8.3244
10.0    0.8836    0.8834            9.4575


## 8. Final evaluation on the official test set

In [8]:
X_test = test_df["text"]
y_test = test_df["category"]

final_model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), max_features=30000)),
    ("clf", LogisticRegression(C=10.0, max_iter=1000, random_state=42)),
])

final_model.fit(X, y)

predictions = final_model.predict(X_test)

print("Official test examples:", len(X_test))
print(f"Test accuracy: {accuracy_score(y_test, predictions):.4f}")
print("Test macro-F1:", f"{f1_score(y_test, predictions, average='macro'):.4f}")

print("\nPer-category performance:")
print(classification_report(y_test, predictions, zero_division=0, digits=3))

Official test examples: 3080
Test accuracy: 0.8929
Test macro-F1: 0.8932

Per-category performance:
                                                  precision    recall  f1-score   support

                           Refund_not_showing_up      0.950     0.950     0.950        40
                                activate_my_card      0.950     0.950     0.950        40
                                       age_limit      1.000     1.000     1.000        40
                         apple_pay_or_google_pay      1.000     1.000     1.000        40
                                     atm_support      0.923     0.900     0.911        40
                                automatic_top_up      1.000     0.900     0.947        40
         balance_not_updated_after_bank_transfer      0.620     0.775     0.689        40
balance_not_updated_after_cheque_or_cash_deposit      0.927     0.950     0.938        40
                         beneficiary_not_allowed      0.950     0.950     0.950        40

## 9. Train and save the final model

In [9]:
print(f"Training examples: {len(train_df)}")
print(f"Intent categories: {y.nunique()}")

model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 1), max_features=30000)),
    ("classifier", LogisticRegression(C=10.0, max_iter=1000, random_state=42)),
])

model.fit(X, y)

print("Model training completed.")

model_path = "ticket_intent_model.joblib"
joblib.dump(model, model_path)
print(f"Trained model saved to: {model_path}")

Training examples: 10003
Intent categories: 77
Model training completed.
Trained model saved to: ticket_intent_model.joblib


## 10. Explain a single prediction

In [10]:
model = joblib.load("ticket_intent_model.joblib")

vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

# Change this text to explain any message.
text = "My card was swallowed by the ATM and I need it back"

if not text.strip():
    raise SystemExit("Please enter a non-empty message.")

X_input = vectorizer.transform([text])

scores = classifier.decision_function(X_input)[0]
ranked_indices = np.argsort(scores)[::-1]

predicted_idx = ranked_indices[0]
runner_up_idx = ranked_indices[1]

predicted_intent = classifier.classes_[predicted_idx]
runner_up_intent = classifier.classes_[runner_up_idx]

print("\nPrediction analysis")
print("-" * 50)
print(f"Message: {text}")
print(f"Predicted intent: {predicted_intent}")
print(f"Runner-up intent: {runner_up_intent}")
print(f"Predicted class score: {scores[predicted_idx]:.4f}")
print(f"Runner-up class score: {scores[runner_up_idx]:.4f}")

# Positive contribution favors the predicted intent,
# negative contribution favors the runner-up.
weight_difference = classifier.coef_[predicted_idx] - classifier.coef_[runner_up_idx]

feature_names = vectorizer.get_feature_names_out()
feature_values = X_input.toarray()[0]

contributions = feature_values * weight_difference

positive_indices = np.argsort(contributions)[::-1]
negative_indices = np.argsort(contributions)

print(f"\nWords/features favoring {predicted_intent}:")
count = 0
for idx in positive_indices:
    if contributions[idx] <= 0:
        break
    if feature_values[idx] > 0:
        print(f"  {feature_names[idx]}: +{contributions[idx]:.4f}")
        count += 1
        if count == 10:
            break

print(f"\nWords/features favoring {runner_up_intent}:")
count = 0
for idx in negative_indices:
    if contributions[idx] >= 0:
        break
    if feature_values[idx] > 0:
        print(f"  {feature_names[idx]}: {contributions[idx]:.4f}")
        count += 1
        if count == 10:
            break


Prediction analysis
--------------------------------------------------
Message: My card was swallowed by the ATM and I need it back
Predicted intent: card_swallowed
Runner-up intent: reverted_card_payment?
Predicted class score: 9.7365
Runner-up class score: 3.6886

Words/features favoring card_swallowed:
  atm: +3.6173
  swallowed: +2.4698
  card: +0.6042
  need: +0.4820
  and: +0.2636
  the: +0.2623
  by: +0.1339
  my: +0.0687

Words/features favoring reverted_card_payment?:
  was: -0.4791
  back: -0.3185
  it: -0.3096
